# 02. Recuperación y contraste de explicitud

Las cuatro estrategias se comparan sobre los mismos rankings y qrels de la campaña final. Se recalculan las métricas con los scripts de la entrega y se contrastan los CSV con las referencias publicadas.

Run All crea un destino nuevo para este análisis. No ejecuta los modelos de recuperación ni modifica sus rankings.

In [ ]:
from pathlib import Path
import sys
import json
import pandas as pd
from IPython.display import display, Markdown

# Jupyter puede abrir el cuaderno desde la raíz técnica o desde notebooks/.
HERE = Path.cwd().resolve()
SUPPORT = next(p / 'notebooks' for p in (HERE, *HERE.parents)
               if (p / 'notebooks/notebook_support.py').is_file())
sys.path.insert(0, str(SUPPORT))
from notebook_support import find_root, read_jsonl, new_run, run_script, compare_csv

ROOT = find_root()
pd.set_option('display.max_columns', 18)
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_colwidth', 90)

In [ ]:
IR = ROOT / 'outputs/local_runs/explicitness_v3'
REFERENCE = ROOT / 'outputs/local_runs/academic_corrections_20260930/retrieval'
RUN = new_run('recuperacion')
print('Salidas:', RUN.relative_to(ROOT))

## Candidatos y orden

BM25, dense, RRF y reranking tienen diez candidatos por formulación. En esta campaña, el reranker cambia el orden de los candidatos de RRF, no el conjunto de diez fragmentos. Por eso Recall@10 y Precision@10 deben coincidir entre ambos, aunque las métricas de orden puedan cambiar.

Algunos registros de ranking conservan el estado de anotación que existía al crearlos. La referencia vigente es qrels_final_v3.jsonl, no aquel campo histórico.

In [ ]:
runs = {method: pd.DataFrame(read_jsonl(IR / 'runs' / f'explicitness_v3_{method}.jsonl'))
        for method in ['bm25', 'dense', 'rrf', 'rerank']}
for method, frame in runs.items():
    assert len(frame) == 520 and frame['qid'].nunique() == 52
    assert frame.groupby('qid').size().eq(10).all()
    assert not frame.duplicated(['qid', 'chunk_id']).any()
rrf_sets = runs['rrf'].groupby('qid')['chunk_id'].agg(set)
rerank_sets = runs['rerank'].groupby('qid')['chunk_id'].agg(set)
assert all(rrf_sets[qid] == rerank_sets[qid] for qid in rrf_sets.index)
display(pd.DataFrame([{'estrategia': method, 'formulaciones': frame['qid'].nunique(), 'filas_ranking': len(frame)}
                      for method, frame in runs.items()]))

## Recálculo de las métricas

El evaluador utiliza relevancia estricta para las métricas binarias e ideal global para nDCG. La tabla que se muestra después procede del directorio recién generado.

In [ ]:
run_script('scripts/evaluate_retrieval_v3.py', '--run-dir', IR,
           '--qrels', IR / 'qrels_final_v3.jsonl', '--methods', 'bm25,dense,rrf,rerank',
           '--output-dir', RUN / 'retrieval', log_dir=RUN)
checked_ir = compare_csv(RUN / 'retrieval', REFERENCE / 'evaluation')
metrics = pd.read_csv(RUN / 'retrieval/aggregate_metrics.csv')
display(metrics)

## Diferencias explícita–implícita

El contraste de recuperación calcula **explícita menos implícita** dentro de cada pregunta base. Un valor positivo indica mejor resultado con la formulación explícita. El test de signos utiliza la dirección de las diferencias y descarta los empates; se muestran sus recuentos y el ajuste Holm junto al valor p.

In [ ]:
run_script('scripts/paired_explicitness_v3.py', '--metrics-dir', RUN / 'retrieval',
           '--historical-metrics-dir', IR / 'retrieval_metrics_v3',
           '--output-dir', RUN / 'paired', log_dir=RUN)
checked_paired = compare_csv(RUN / 'paired', REFERENCE / 'paired')
paired = pd.read_csv(RUN / 'paired/paired_explicitness_v3.csv')
display(paired.loc[paired['k'] == 10,
    ['method', 'metric', 'mean_diff', 'pos', 'neg', 'ties', 'n_effective', 'p_sign', 'p_holm']])

La sensibilidad restringe el análisis a las bases con evidencia de grado 2. Los archivos de diferencias por pregunta permiten revisar si la media depende de pocos casos; las diferencias entre efectos de métodos son descriptivas.

In [ ]:
display(pd.read_csv(RUN / 'paired/paired_grade2_eligible.csv'))
display(pd.read_csv(RUN / 'paired/between_method_effect_differences.csv'))

## Comparación con las referencias

La comprobación exige el mismo conjunto de CSV y el mismo contenido binario. No compara los tiempos de ejecución ni vuelve a ejecutar inferencia.

In [ ]:
checked = pd.DataFrame([dict(etapa='recuperacion', **row) for row in checked_ir] +
                       [dict(etapa='explicitud', **row) for row in checked_paired])
assert checked['identico'].all()
display(checked)

Una mejora de ordenación no implica mayor cobertura. Estas métricas describen la recuperación de evidencia; la calidad de las respuestas generadas se examina por separado.